In [22]:
import os
import json
import requests

# --- CONFIGURATION & ENV SETUP ---
# MAAS_URL is the MaaS hostname, e.g. maas.apps.rosa.<cluster>.<domain>
MAAS_URL = os.environ["MAAS_URL"]
# The MaaS subscription to create the key for
MAAS_SUBSCRIPTION = os.environ.get("MAAS_SUBSCRIPTION", "gpt-oss-20b-premium")

# In an OpenShift AI workbench, authenticate as the workbench's ServiceAccount
# (its token is mounted into the pod). Set OC_TOKEN to use another token.
SA_TOKEN_FILE = "/var/run/secrets/kubernetes.io/serviceaccount/token"
if os.environ.get("OC_TOKEN"):
    OC_TOKEN = os.environ["OC_TOKEN"]
else:
    with open(SA_TOKEN_FILE) as f:
        OC_TOKEN = f.read().strip()

print(f"Configured MAAS_URL: {MAAS_URL}")


# -------------------------------------------------------------------------
# STEP 1: Generate MaaS API Key
# -------------------------------------------------------------------------
print("\n[Step 1/3] Generating API Key...")
keygen_url = f"https://{MAAS_URL}/maas-api/v1/api-keys"

headers_step1 = {
    "Authorization": f"Bearer {OC_TOKEN}",
    "Content-Type": "application/json"
}
payload_step1 = {
    "name": "test-key", 
    "subscription": MAAS_SUBSCRIPTION, 
    "expiresIn": "1h"
}

response_step1 = requests.post(
    keygen_url, 
    headers=headers_step1, 
    json=payload_step1
)
response_step1.raise_for_status()

# Equivalent to storing the key parsed via jq
api_key = response_step1.json()["key"]
print("Key generated successfully.")


In [23]:
# -------------------------------------------------------------------------
# STEP 2: List Models & Extract Target URL
# -------------------------------------------------------------------------
print("\n[Step 2/3] Querying Available Models...")
models_url = f"https://{MAAS_URL}/v1/models"

headers_step2 = {
    "Authorization": f"Bearer {api_key}"
}

response_step2 = requests.get(
    models_url, 
    headers=headers_step2
)
response_step2.raise_for_status()
models_json = response_step2.json()

print(response_step2.json())

# Use the model named in MAAS_MODEL, or the first one listed. Your key may not
# be authorized for every model in the list.
wanted = os.environ.get("MAAS_MODEL")
models = models_json["data"]
model = next((m for m in models if m["id"] == wanted), None) if wanted else models[0]
if model is None:
    raise ValueError(f"MAAS_MODEL={wanted} is not offered; available: {[m['id'] for m in models]}")
model_url = model["url"]
model_id = model["id"]

print(f"-> Target Model ID: {model_id}")
print(f"-> Extracted Model URL: {model_url}")

In [24]:
# -------------------------------------------------------------------------
# STEP 3: Submit Chat Completion Request
# -------------------------------------------------------------------------
print("\n[Step 3/3] Sending Chat Completion...")
# Model URLs may end in "/"; a double slash in the path returns 404
chat_url = f"{model_url.rstrip('/')}/v1/chat/completions"

headers_step3 = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
payload_step3 = {
    "model": model_id,
    "messages": [{"role": "user", "content": "What is a fish?"}]
}

response_step3 = requests.post(
    chat_url, 
    headers=headers_step3, 
    json=payload_step3
)
response_step3.raise_for_status()

print("\n--- Final API Response ---")
print(json.dumps(response_step3.json(), indent=2))